# EEG_23 — PLV: Phase Locking Value per C0/C1

**Domanda**: Il meccanismo IS di C0/C1 è visibile nella sincronizzazione di fase tra elettrodi (PLV),
dove la PSD non riesce?

**Tre analisi:**
- **A** — C0 vs C1 (tutti i trial): i fenotipi differiscono in PLV? (cross-validazione indipendente)
- **B** — Dentro C0: corretti vs sbagliati (tutti soggetti C0)
- **C** — Dentro C1: corretti vs sbagliati (tutti soggetti C1) — ipotesi F3↔PO8

**Bande**: alpha (8-13 Hz), beta (13-30 Hz), gamma (30-50 Hz)
Delta/theta escluse: trial da 1.5s troppo brevi per stima affidabile della fase.

## §1 — Setup

In [ ]:
import json, logging, re
from collections import defaultdict
from pathlib import Path

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.signal import butter, filtfilt, hilbert
from scipy.stats import mannwhitneyu
from sklearn.metrics import balanced_accuracy_score
from tqdm.auto import tqdm
import mne

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)-8s %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('eeg23')

project_root = Path('/home/daniele_u/miralis-hypergraph-imagined-speech')
FIG_DIR = project_root / 'figures'; FIG_DIR.mkdir(exist_ok=True)
CKPT_13B = project_root / 'models' / 'eeg13b_200e'

N_CHANNELS = 61; N_SAMPLES = 384; N_CLASSES = 4; FS = 256
K_WINDOWS = 8; N_EDGES = 16; D_MODEL = 64; HIDDEN = 128; N_LAYERS = 2; DROPOUT = 0.5
T_WIN = N_SAMPLES // K_WINDOWS; CLUSTER_SCHEME = 'concr4'; DATA_METRIC = 'abs_pcc'
CLUSTER_NAMES = {0: 'C0 Fronto-motor', 1: 'C1 Fronto-occipital'}
PLV_BANDS = {'alpha': (8, 13), 'beta': (13, 30), 'gamma': (30, 50)}
device = torch.device('cpu')

with open(project_root / 'configs' / 'label_schemes' / 'label2idx.json') as f:
    WORD2LABEL = json.load(f)
with open(project_root / 'configs' / 'label_schemes' / f'labelid2cluster_{CLUSTER_SCHEME}.json') as f:
    _raw = json.load(f); label2cluster = {int(k): int(v) for k, v in _raw.items()}
_cd = json.loads((project_root / 'configs' / 'eeg16b_cluster_labels.json').read_text())
SUBJ_CLUSTER = {s: l for s, l in zip(_cd['subj_ids'], _cd['labels'])}

_PAT = re.compile(r'^P(\d+)_S(\d+)$')
subj_sess = defaultdict(lambda: defaultdict(list))
_root = project_root / 'data' / f'hypergraphs_pruned_{DATA_METRIC}'
for p in sorted(_root.rglob('trial_*.pt')):
    m = _PAT.match(p.parent.name)
    if m: subj_sess[int(m.group(1))][int(m.group(2))].append(p)

C0_SUBJ = sorted([s for s, c in SUBJ_CLUSTER.items() if c == 0])
C1_SUBJ = sorted([s for s, c in SUBJ_CLUSTER.items() if c == 1])
ALL_SUBJ = C0_SUBJ + C1_SUBJ
log.info(f'C0={len(C0_SUBJ)} sogg  C1={len(C1_SUBJ)} sogg  totale={len(ALL_SUBJ)}')

CHAN_NAMES_MNE = ['A1','AF7','AF3','Fp1','Fp2','AF4','AF8','A2',
    'F7','F5','F3','F1','F2','F4','F6','F8',
    'FT7','FC5','FC3','FC1','FC2','FC4','FC6','FT8',
    'T7','C5','C3','C1','C2','C4','C6','T8',
    'TP7','CP5','CP3','CP1','CP2','CP4','CP6','TP8',
    'P7','P5','P3','P1','P2','P4','P6','P8',
    'FPz','PO7','PO3','O1','O2','PO4','PO8','Oz',
    'AFz','Fz','FCz','Cz','CPz']
CHAN_IDX = {n: i for i, n in enumerate(CHAN_NAMES_MNE)}

_info = mne.create_info(ch_names=CHAN_NAMES_MNE, sfreq=FS, ch_types='eeg')
_info.set_montage(mne.channels.make_standard_montage('standard_1020'), on_missing='ignore')
HEAD_SCALE = 1.15
log.info('Setup completato.')


## §2 — DHSLP (identico a EEG_22)

In [ ]:
class HGNNConv(nn.Module):
    def __init__(self,in_dim,out_dim):
        super().__init__()
        self.weight=nn.Parameter(torch.empty(in_dim,out_dim))
        self.bias=nn.Parameter(torch.zeros(out_dim))
        nn.init.xavier_uniform_(self.weight)
    def forward(self,x,H):
        Dv=H.sum(dim=2,keepdim=True).clamp(min=1e-6)
        De=H.sum(dim=1).clamp(min=1e-6).unsqueeze(2)
        Ht=H.transpose(1,2); x_norm=x/Dv
        return torch.bmm(H,torch.bmm(Ht,x_norm)/De)@self.weight+self.bias

class DHSLP(nn.Module):
    def __init__(self,n_nodes=N_CHANNELS,T_win=T_WIN,K=K_WINDOWS,n_edges=N_EDGES,
                 d_model=D_MODEL,hidden=HIDDEN,n_classes=N_CLASSES,n_layers=N_LAYERS,dropout=DROPOUT):
        super().__init__()
        self.K=K; self.T_win=T_win; self.d_model=d_model
        self.E=nn.Parameter(torch.randn(n_edges,d_model)*0.01)
        self.pos_enc=nn.Parameter(torch.randn(n_nodes,d_model)*0.01)
        self.node_proj=nn.Sequential(nn.Linear(T_win,d_model),nn.LayerNorm(d_model),nn.ELU())
        dims=[d_model]+[hidden]*n_layers
        self.convs=nn.ModuleList([HGNNConv(dims[i],dims[i+1]) for i in range(n_layers)])
        self.bns=nn.ModuleList([nn.BatchNorm1d(hidden) for _ in range(n_layers)])
        self.drop=nn.Dropout(dropout); self.clf=nn.Linear(hidden,n_classes)
    def build_dynamic_H(self,feat):
        return torch.softmax(torch.matmul(feat,self.E.T)/(self.d_model**0.5),dim=-1)
    def forward(self,x):
        B,N,T=x.shape
        wins=x.reshape(B,N,self.K,self.T_win).permute(0,2,1,3).reshape(B*self.K,N,self.T_win)
        feat=self.node_proj(wins)+self.pos_enc.unsqueeze(0)
        H=self.build_dynamic_H(feat); h=feat
        for conv,bn in zip(self.convs,self.bns):
            h=conv(h,H); h=bn(h.reshape(-1,h.shape[-1])).reshape(h.shape); h=F.elu(h); h=self.drop(h)
        return self.clf(h.mean(dim=1).reshape(B,self.K,-1).mean(dim=1))

def load_model(sid):
    p = CKPT_13B / f'P{sid:03d}.pt'
    if not p.exists(): return None
    ck = torch.load(p, map_location=device, weights_only=False)
    m = DHSLP().to(device); m.load_state_dict(ck['state_dict']); m.eval(); return m

log.info('DHSLP pronto.')


## §3 — Inference su TUTTI i soggetti C0+C1

Raccoglie: segnale grezzo, cluster, correct flag, per ogni trial.

In [ ]:
def run_inference_all(subj_ids):
    """Inference su tutti i soggetti. Restituisce lista di record con segnale grezzo incluso."""
    records = []
    for sid in tqdm(subj_ids, desc='Inference'):
        cl = SUBJ_CLUSTER.get(sid)
        if cl is None: continue
        model = load_model(sid)
        if model is None: continue
        with torch.no_grad():
            for sess_id, paths in sorted(subj_sess[sid].items()):
                for p in paths:
                    d = torch.load(p, weights_only=False)
                    x = d['x'].float()  # (61, 384)
                    y_word = int(d['y'].squeeze()) if isinstance(d['y'], torch.Tensor) else int(d['y'])
                    y_true = label2cluster.get(y_word)
                    if y_true is None: continue
                    x_norm = (x - x.mean(1, keepdim=True)) / (x.std(1, keepdim=True) + 1e-6)
                    y_pred = int(F.softmax(model(x_norm.unsqueeze(0)), dim=1).squeeze().cpu().numpy().argmax())
                    records.append({
                        'subj_id': sid, 'cluster': cl, 'sess_id': sess_id,
                        'y_true': y_true, 'y_pred': y_pred,
                        'correct': int(y_true == y_pred),
                        'x': x.numpy()  # (61, 384) — segnale grezzo per PLV
                    })
        del model

    log.info(f'Record totali: {len(records)}')
    c0 = [r for r in records if r['cluster']==0]
    c1 = [r for r in records if r['cluster']==1]
    log.info(f'C0: {len(c0)} trial ({sum(r["correct"] for r in c0)} corretti)')
    log.info(f'C1: {len(c1)} trial ({sum(r["correct"] for r in c1)} corretti)')
    return records

ALL_RECORDS = run_inference_all(ALL_SUBJ)


## §4 — PLV computation

Per ogni trial: bandpass → Hilbert → fase → PLV matrix (61×61) per banda.

In [ ]:
def bandpass_filter(x, lo, hi, fs=FS, order=4):
    """x: (61, T) → filtered (61, T)"""
    nyq = fs / 2
    b, a = butter(order, [lo / nyq, hi / nyq], btype='band')
    return filtfilt(b, a, x, axis=1)

def compute_plv_matrix(x_np, band):
    """
    x_np: (61, 384)
    band: (lo, hi) Hz
    Returns PLV matrix (61, 61) — valori in [0, 1]
    """
    lo, hi = band
    x_filt = bandpass_filter(x_np, lo, hi)          # (61, T)
    analytic = hilbert(x_filt, axis=1)               # (61, T) complex
    phase = np.exp(1j * np.angle(analytic))          # unit complex
    # PLV[i,j] = |mean_t( phase[i,t] * conj(phase[j,t]) )|
    plv = np.abs(phase @ phase.conj().T) / x_np.shape[1]
    np.fill_diagonal(plv, 0.0)                       # no self-coupling
    return plv.astype(np.float32)

def compute_plv_all_bands(x_np):
    return {band: compute_plv_matrix(x_np, lo_hi) for band, lo_hi in PLV_BANDS.items()}

# Test su un trial
_test = ALL_RECORDS[0]
_plv = compute_plv_all_bands(_test['x'])
log.info(f'PLV shape per banda: {_plv["alpha"].shape}  range alpha: [{_plv["alpha"].min():.3f}, {_plv["alpha"].max():.3f}]')


## §5 — Pre-calcolo PLV per tutti i trial

Calcola e archivia la matrice PLV per ogni trial (~50-70K trial — può richiedere 15-20 min).

In [ ]:
log.info('Pre-calcolo PLV per tutti i trial...')
for r in tqdm(ALL_RECORDS, desc='PLV'):
    r['plv'] = compute_plv_all_bands(r['x'])
    del r['x']  # libera memoria (segnale grezzo non più necessario)
log.info('PLV calcolato per tutti i trial.')


## §6 — Funzioni di analisi e visualizzazione

In [ ]:
def plv_diff_analysis(records_a, records_b, label_a='A', label_b='B'):
    """
    Confronta PLV di due gruppi di trial per ogni banda e coppia di elettrodi.
    Ritorna: per ogni banda → array (61,61) di Cohen's d e p-values.
    """
    results = {}
    for band in PLV_BANDS:
        plv_a = np.stack([r['plv'][band] for r in records_a])  # (N_a, 61, 61)
        plv_b = np.stack([r['plv'][band] for r in records_b])  # (N_b, 61, 61)
        n_ch = plv_a.shape[1]
        cohd = np.zeros((n_ch, n_ch), dtype=np.float32)
        pval = np.ones((n_ch, n_ch), dtype=np.float32)
        # Solo upper triangle (matrice simmetrica)
        for i in range(n_ch):
            for j in range(i+1, n_ch):
                a_ij = plv_a[:, i, j]
                b_ij = plv_b[:, i, j]
                _, p = mannwhitneyu(a_ij, b_ij, alternative='two-sided')
                ps = np.sqrt((a_ij.std()**2 + b_ij.std()**2) / 2 + 1e-12)
                d = (a_ij.mean() - b_ij.mean()) / ps
                cohd[i, j] = cohd[j, i] = d
                pval[i, j] = pval[j, i] = p
        np.fill_diagonal(pval, 1.0); np.fill_diagonal(cohd, 0.0)
        nsig = int((pval < 0.05).sum() // 2)
        log.info(f'  {band}: {nsig}/1830 coppie sig ({label_a} vs {label_b})')
        results[band] = {'cohd': cohd, 'pval': pval, 'nsig': nsig}
    return results

def per_electrode_summary(cohd_mat, pval_mat, method='mean_abs'):
    """
    Collassa la matrice (61,61) in un vettore per elettrodo (61,).
    method='mean_abs': media del |d| su tutte le coppie significative
    """
    sig = pval_mat < 0.05
    out = np.zeros(N_CHANNELS)
    for i in range(N_CHANNELS):
        pairs = sig[i, :] & (np.arange(N_CHANNELS) != i)
        if pairs.sum() > 0:
            out[i] = np.abs(cohd_mat[i, pairs]).mean()
    return out

def plot_plv_analysis(results, title, fname, vlim=0.3):
    """3 plot per riga (alpha/beta/gamma): heatmap + topomap per-elettrodo."""
    bands = list(PLV_BANDS.keys())
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    fig.patch.set_facecolor('white')
    fig.suptitle(title, fontsize=13, color='black')

    for col, band in enumerate(bands):
        cohd = results[band]['cohd']
        pval = results[band]['pval']
        nsig = results[band]['nsig']

        # Row 0: heatmap 61×61
        ax = axes[0, col]; ax.set_facecolor('white')
        im = ax.imshow(cohd, cmap='RdBu_r', vmin=-vlim, vmax=vlim, aspect='auto')
        ax.set_title(f"{band.capitalize()} — {nsig}/1830 coppie sig\nCohen's d (A−B)", fontsize=9, color='black')
        ax.set_xlabel('Elettrodo', color='black', fontsize=8)
        ax.set_ylabel('Elettrodo', color='black', fontsize=8)
        ax.tick_params(colors='black', labelsize=6)
        plt.colorbar(im, ax=ax, fraction=0.046)

        # Row 1: topomap per-elettrodo
        ax = axes[1, col]; ax.set_facecolor('white')
        elec_vals = per_electrode_summary(cohd, pval)
        mask = elec_vals > 0
        im2, _ = mne.viz.plot_topomap(
            elec_vals, _info, axes=ax, cmap='Reds', vlim=(0, vlim),
            mask=mask, mask_params=dict(marker='o', markerfacecolor='k',
                                         markeredgecolor='k', markersize=5, linewidth=0),
            show=False, sphere=HEAD_SCALE)
        ax.set_title(f'{band.capitalize()} — |d| medio per elettrodo\n(punti neri = ha coppie sig)', fontsize=8, color='black')
        plt.colorbar(im2, ax=ax, fraction=0.046, label='mean |d|')

    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    log.info(f'Salvato: {fname}')

def plot_connectome(results, band, title, fname, top_n=30, vlim=0.25):
    """Disegna le top_n coppie più significative come connettoma."""
    cohd = results[band]['cohd']
    pval = results[band]['pval']

    # Prendi le top_n coppie per |d| tra quelle significative
    sig_mask = pval < 0.05
    pairs = [(i, j, cohd[i,j], pval[i,j])
             for i in range(N_CHANNELS)
             for j in range(i+1, N_CHANNELS)
             if sig_mask[i,j]]
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    top_pairs = pairs[:top_n]

    if not top_pairs:
        log.warning(f'Nessuna coppia significativa per {band}')
        return

    fig, ax = plt.subplots(1, 1, figsize=(7, 7))
    fig.patch.set_facecolor('white')

    # Topomap di base (valori zero, solo per il layout)
    mne.viz.plot_topomap(np.zeros(N_CHANNELS), _info, axes=ax,
                          show=False, sphere=HEAD_SCALE,
                          cmap='Greys', vlim=(0, 1))

    # Posizioni degli elettrodi nel piano topomap
    from mne.viz.topomap import _get_pos_outlines
    pos, _ = _get_pos_outlines(_info, picks=None, sphere=HEAD_SCALE)
    pos_arr = pos  # (61, 2)

    norm = plt.Normalize(vmin=-vlim, vmax=vlim)
    cmap = plt.cm.RdBu_r

    for i, j, d, p in top_pairs:
        x_vals = [pos_arr[i, 0], pos_arr[j, 0]]
        y_vals = [pos_arr[i, 1], pos_arr[j, 1]]
        ax.plot(x_vals, y_vals, color=cmap(norm(d)),
                lw=1.5 + 2*abs(d)/vlim, alpha=0.7, zorder=3)

    # Colorbar
    sm = plt.cm.ScalarMappable(cmap=cmap, norm=norm)
    plt.colorbar(sm, ax=ax, fraction=0.046, label="Cohen's d")

    ax.set_title(f"{title}\n{band.capitalize()} — top {len(top_pairs)} coppie sig (p<0.05)",
                 fontsize=10, color='black')
    plt.tight_layout()
    plt.savefig(FIG_DIR / fname, dpi=150, bbox_inches='tight', facecolor='white')
    plt.close()
    log.info(f'Salvato: {fname}')

log.info('Funzioni pronte.')


## §7 — Analisi A: C0 vs C1 (tutti i trial)

I due fenotipi differiscono in PLV? Cross-validazione indipendente da abs_pcc.

In [ ]:
log.info('=== ANALISI A: C0 vs C1 (tutti i trial) ===')
c0_all = [r for r in ALL_RECORDS if r['cluster'] == 0]
c1_all = [r for r in ALL_RECORDS if r['cluster'] == 1]
log.info(f'C0: {len(c0_all)} trial   C1: {len(c1_all)} trial')

results_A = plv_diff_analysis(c0_all, c1_all, 'C0', 'C1')
plot_plv_analysis(results_A,
    title='Analisi A — PLV: C0 vs C1 (tutti i trial, tutti i soggetti)\n'
          'rosso=C0>C1  blu=C0<C1  punti neri=p<0.05',
    fname='eeg23_A_c0_vs_c1.png')

# Connectome per alpha e gamma
for band in ['alpha', 'gamma']:
    plot_connectome(results_A, band,
        title='C0 vs C1',
        fname=f'eeg23_A_connectome_{band}.png')


## §8 — Analisi B: dentro C0, corretti vs sbagliati (tutti sogg C0)

In [ ]:
log.info('=== ANALISI B: C0 corretti vs sbagliati ===')
c0_ok = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==1]
c0_no = [r for r in ALL_RECORDS if r['cluster']==0 and r['correct']==0]
log.info(f'C0 corretti: {len(c0_ok)}   C0 sbagliati: {len(c0_no)}')

results_B = plv_diff_analysis(c0_ok, c0_no, 'C0-ok', 'C0-no')
plot_plv_analysis(results_B,
    title='Analisi B — PLV: C0 corretti vs sbagliati (tutti sogg C0)\n'
          'rosso=corretti>sbagliati  blu=corretti<sbagliati',
    fname='eeg23_B_c0_correct_vs_wrong.png')

for band in ['alpha', 'gamma']:
    plot_connectome(results_B, band,
        title='C0 corretti vs sbagliati',
        fname=f'eeg23_B_connectome_{band}.png')


## §9 — Analisi C: dentro C1, corretti vs sbagliati (tutti sogg C1)

**Ipotesi principale**: PLV F3↔PO8 più alto nei trial corretti C1.

In [ ]:
log.info('=== ANALISI C: C1 corretti vs sbagliati ===')
c1_ok = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==1]
c1_no = [r for r in ALL_RECORDS if r['cluster']==1 and r['correct']==0]
log.info(f'C1 corretti: {len(c1_ok)}   C1 sbagliati: {len(c1_no)}')

results_C = plv_diff_analysis(c1_ok, c1_no, 'C1-ok', 'C1-no')
plot_plv_analysis(results_C,
    title='Analisi C — PLV: C1 corretti vs sbagliati (tutti sogg C1)\n'
          'rosso=corretti>sbagliati  blu=corretti<sbagliati',
    fname='eeg23_C_c1_correct_vs_wrong.png')

for band in ['alpha', 'gamma']:
    plot_connectome(results_C, band,
        title='C1 corretti vs sbagliati',
        fname=f'eeg23_C_connectome_{band}.png')


## §10 — Spotlight F3↔PO8

Analisi dedicata alla coppia hub di C1 su tutte le bande e tutti i gruppi.

In [ ]:
F3_IDX  = CHAN_IDX['F3']
PO8_IDX = CHAN_IDX['PO8']

log.info(f'F3={F3_IDX}  PO8={PO8_IDX}')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.patch.set_facecolor('white')
fig.suptitle('Spotlight F3↔PO8 — PLV per banda e gruppo', fontsize=12, color='black')

groups = [
    (c0_all,  'C0 (tutti)',     '#4A90E2'),
    (c1_all,  'C1 (tutti)',     '#FF8C42'),
    (c0_ok,   'C0 corretti',   '#52B788'),
    (c0_no,   'C0 sbagliati',  '#E63946'),
    (c1_ok,   'C1 corretti',   '#52B788'),
    (c1_no,   'C1 sbagliati',  '#E63946'),
]

for ax, band in zip(axes, PLV_BANDS):
    ax.set_facecolor('white')
    data_per_group = []
    labels = []
    colors_box = []
    for records, label, color in groups:
        vals = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in records])
        data_per_group.append(vals)
        labels.append(label)
        colors_box.append(color)

    vp = ax.violinplot(data_per_group, positions=range(len(groups)), showmedians=True)
    for body, color in zip(vp['bodies'], colors_box):
        body.set_facecolor(color); body.set_alpha(0.6)
    vp['cmedians'].set_colors('black')

    # Mann-Whitney C1 ok vs C1 no
    _, p_c1 = mannwhitneyu(data_per_group[4], data_per_group[5], alternative='two-sided')
    # Mann-Whitney C0 ok vs C0 no
    _, p_c0 = mannwhitneyu(data_per_group[2], data_per_group[3], alternative='two-sided')

    ax.set_xticks(range(len(groups)))
    ax.set_xticklabels(labels, rotation=30, ha='right', fontsize=8, color='black')
    ax.set_title(f'{band.capitalize()}\nC0 ok vs no: p={p_c0:.4f}\nC1 ok vs no: p={p_c1:.4f}',
                 fontsize=9, color='black')
    ax.set_ylabel('PLV F3↔PO8', color='black', fontsize=9)
    ax.tick_params(colors='black')
    for sp in ax.spines.values(): sp.set_edgecolor('#CCCCCC')

plt.tight_layout()
plt.savefig(FIG_DIR / 'eeg23_spotlight_F3_PO8.png', dpi=150, bbox_inches='tight', facecolor='white')
plt.close()
log.info('Salvato: eeg23_spotlight_F3_PO8.png')


## §11 — Verdict

In [ ]:
print('=' * 60)
print('EEG_23 — VERDICT')
print('=' * 60)
print()
print('Analisi A — C0 vs C1 (tutti i trial):')
for band, res in results_A.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('Analisi B — C0 corretti vs sbagliati:')
for band, res in results_B.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('Analisi C — C1 corretti vs sbagliati:')
for band, res in results_C.items():
    print(f'  {band:8s}: {res["nsig"]:4d}/1830 coppie sig')

print()
print('F3-PO8 specifico:')
for band in PLV_BANDS:
    c1_ok_vals  = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in c1_ok])
    c1_no_vals  = np.array([r['plv'][band][F3_IDX, PO8_IDX] for r in c1_no])
    _, p = mannwhitneyu(c1_ok_vals, c1_no_vals, alternative='two-sided')
    ps = np.sqrt((c1_ok_vals.std()**2 + c1_no_vals.std()**2)/2 + 1e-12)
    d  = (c1_ok_vals.mean() - c1_no_vals.mean()) / ps
    print(f'  C1 {band:8s} F3-PO8: d={d:+.4f}  p={p:.4f}  ok_mean={c1_ok_vals.mean():.4f}  no_mean={c1_no_vals.mean():.4f}')
print('=' * 60)
